# Пример: dense GAN и RGB DCGAN для генерации лиц

Этот пример сопоставляет две архитектуры генеративно-состязательной
сети на цветных лицевых изображениях LFW. Обе модели получают один и
тот же латентный вектор, обучаются на одном детерминированном срезе
данных и различаются только способом обработки пространственной
структуры изображения.

Dense GAN рассматривает лицо как длинный вектор из $64\cdot64\cdot3$
чисел. DCGAN сохраняет двумерную сетку пикселей и использует свёртки с
общими весами. Поэтому сопоставление показывает не «абсолютного
победителя», а влияние архитектурного предположения на обучение.

Выполняйте ячейки сверху вниз. Метки персон из LFW не являются целью
сети и не передаются в функцию потерь: они применяются лишь для
проверки корректности загрузки лиц.

## История одного мини-пакета

Представьте художника и проверяющего. Художник получает $B$ карточек
со 100 случайными числами и создаёт $B$ картинок лиц. Проверяющий
получает эти картинки вместе с $B$ настоящими лицами и ставит на
каждой отметку «больше похоже на настоящее» или «больше похоже на
рисунок». Сначала поправляются веса проверяющего, потом — веса
художника. Эти два шага повторяются для каждого мини-пакета в
`train_gan`.

Идентификатор человека — всего лишь подпись под примером. GAN не
учится узнавать людей и не связывает конкретный шум с человеком: она
старается освоить общий вид изображений из набора.

### Словарь перед первой реализацией

Параметры — числа в слоях, которые можно менять. Прямой проход —
создание картинки и её проверка. Потеря — одно число, показывающее
расхождение текущего ответа с целью. Градиент — подсказка, в какую
сторону изменить каждый параметр. `GradientTape` находит подсказки, а
Adam применяет небольшую поправку к весам выбранной сети.

## Этап 1. Конфигурация и два профиля вычислений

**Входы:** настройки воспроизводимости и выбранный профиль.
**Действие:** профиль CPU использует 2 500 лиц и 80 эпох. Профиль
Arc предназначен для отдельного PyTorch/XPU
ноутбука; здесь TensorFlow работает на CPU.
**Выход:** определены форма реального изображения $(B,64,64,3)$,
размерность шума $(B,100)$ и каталог артефактов.
**Проверка:** изменяйте только значение `PROFILE`, не отдельные
константы внутри цикла обучения.

`(B,64,64,3)` означает: в пачке `B` картинок, каждая имеет 64 строки,
64 столбца и три цветовых канала RGB. `(B,100)` — пачка карточек шума,
а `(B,1)` — один ответ проверяющего на каждую картинку. `B` — размер
пачки, а не часть формы одного лица.

In [ ]:
from __future__ import annotations

# Path хранит только артефакты запуска, time — длительность двух
# экспериментов; ни один из них не участвует в графе производных.
from pathlib import Path
import time

# В ноутбуке нужен встроенный вывод, чтобы Matplotlib не открывал
# отдельное GUI-окно при выполнении учебных ячеек.
from IPython import get_ipython
shell = get_ipython()
if shell is not None:
    shell.run_line_magic('matplotlib', 'inline')
# NumPy хранит срез LFW, TensorFlow строит модели и вычисляет градиенты.
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.datasets import fetch_lfw_people

# Фиксируем выбор лиц, начальные веса и выборки шума.
SEED = 42
PROFILE = 'cpu'  # CPU: 2 500 лиц, 80 эпох; профиль Arc выполняется в PyTorch-ноутбуке.
PROFILES = {
    'cpu': {'max_examples': 2_500, 'epochs': 80, 'batch_size': 32},
    'arc': {'max_examples': 8_000, 'epochs': 120, 'batch_size': 128},
}
# Все три связанные константы берём из одного профиля, чтобы нельзя
# было случайно сочетать CPU-объём данных с числом эпох Arc.
config = PROFILES[PROFILE]
IMAGE_SIZE = 64
CHANNELS = 3
NOISE_DIM = 100
REAL_LABEL = 0.9
SNAPSHOT_EVERY = 5
DATA_CACHE = Path('artifacts/scikit_learn')
ARTIFACT_DIR = Path('artifacts/lfw_face_gan_tensorflow') / PROFILE
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Выполняется до создания слоёв и первого tf.random.normal.
tf.keras.utils.set_random_seed(SEED)
print(f'TensorFlow {tf.__version__}; устройство: CPU; профиль: {PROFILE}')
print('Ожидаемые формы: z=(B,100), x=(B,64,64,3), D(x)=(B,1).')

## Этап 2. Загрузка и подготовка лиц LFW

**Вход:** изображения LFW в цветовом представлении и идентификаторы
персон. **Действие:** фиксированно перемешать записи, выбрать срез,
изменить размер до $64\times64$, перевести уже нормированные значения
из $[0,1]$ в $[-1,1]$. **Выход:** матрица реальных изображений
$X_{\rm real}\in[-1,1]^{N\times64\times64\times3}$.

Идентификатор человека не является ни условием генератора, ни целевой
меткой дискриминатора. Дискриминатор различает только происхождение
картинки: «из набора» или «создана генератором».

### Почему лица нельзя показывать чёрными

Использованный API уже хранит LFW как числа от 0 до 1: 0 — чёрный,
1 — светлый пиксель. Генератор заканчивается `tanh` и выдаёт числа от
-1 до 1. Поэтому реальные лица умножаются на 2 и из результата
вычитается 1: оба источника оказываются на одной шкале. Если разделить
эти данные ещё раз на 255 или 127.5, почти всё станет близко к -1, а
на экране появятся чёрные прямоугольники вместо лиц.

В `dataset` кладутся только изображения. `person_ids` остаются для
подписи диагностической сетки, но не проходят в G, D или функцию потерь.

In [ ]:
# LFW возвращает RGB-массивы и ID персон. ID нужны ниже лишь для
# диагностики: это не условие G и не мишень D.
lfw = fetch_lfw_people(
    data_home=str(DATA_CACHE), color=True, resize=0.5,
    min_faces_per_person=0, download_if_missing=True,
)
# Перестановка задаёт воспроизводимый, но не исходно упорядоченный срез.
order = np.random.default_rng(SEED).permutation(len(lfw.images))
selected = order[:config['max_examples']]
# API уже возвращает float32 в [0,1], поэтому дополнительное деление
# на 255/127.5 здесь было бы ошибкой и сделало бы лица чёрными.
images_unit = lfw.images[selected].astype('float32')
person_ids = lfw.target[selected]
# fetch_lfw_people уже возвращает float32 в диапазоне [0, 1].
assert 0.0 <= float(images_unit.min()) <= float(images_unit.max()) <= 1.0
print('LFW до нормализации: min={:.4f}, q01={:.4f}, median={:.4f}, q99={:.4f}, max={:.4f}'.format(
    *np.quantile(images_unit, [0.00, 0.01, 0.50, 0.99, 1.00])
))
# resize сохраняет соглашение TensorFlow NHWC: (N,H,W,C).
images_unit = tf.image.resize(images_unit, (IMAGE_SIZE, IMAGE_SIZE), antialias=True).numpy()
# Выход G завершается tanh, поэтому приводим реальные лица к [-1,1].
x_real = 2.0 * images_unit - 1.0

assert x_real.shape == (config['max_examples'], IMAGE_SIZE, IMAGE_SIZE, CHANNELS)
assert person_ids.shape == (config['max_examples'],)
assert np.isfinite(x_real).all()
assert -1.0 <= float(x_real.min()) <= float(x_real.max()) <= 1.0
print('LFW после нормализации: min={:.4f}, q01={:.4f}, median={:.4f}, q99={:.4f}, max={:.4f}'.format(
    *np.quantile(x_real, [0.00, 0.01, 0.50, 0.99, 1.00])
))

# Dataset несёт только изображения. shuffle меняет мини-пакеты между
# эпохами, batch формирует (B,64,64,3), prefetch готовит следующий.
dataset = (
    tf.data.Dataset.from_tensor_slices(x_real)
    .shuffle(len(x_real), seed=SEED, reshuffle_each_iteration=True)
    .batch(config['batch_size'], drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)

fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for axis, image, person_id in zip(axes.flat, x_real[:12], person_ids[:12]):
    axis.imshow(np.clip((image + 1.0) / 2.0, 0.0, 1.0), vmin=0.0, vmax=1.0)
    axis.set_title(f'id={int(person_id)}', fontsize=8)
    axis.axis('off')
fig.suptitle('LFW: идентификаторы показаны только для диагностики')
plt.tight_layout()
plt.show()
print('Мини-пакет реальных изображений:', tuple(next(iter(dataset)).shape))

## Этап 3. Две архитектуры и диагностический проход

![Два игрока GAN: генератор создаёт изображение из шума, а дискриминатор возвращает логит его происхождения.](assets/figures/gan_adversarial_architecture.png)

Обе реализации завершают дискриминатор одним **логитом**, то есть
неограниченной оценкой до sigmoid. Функция потерь вычисляет BCE прямо
по логитам, что численно устойчивее отдельного sigmoid. Генератор
возвращает RGB-изображение через `tanh`, поэтому его диапазон совпадает
с диапазоном подготовленных реальных лиц.

**Проверка:** каждая модель должна преобразовать шум $(B,100)$ в
$(B,64,64,3)$, а дискриминатор — в $(B,1)$.

### Два способа собрать лицо

Dense GAN воспринимает лицо как длинный список из 12 288 чисел. Он не
знает заранее, что соседние числа отвечают за соседние пиксели. DCGAN
строит небольшую карту $4\times4$, а затем увеличивает её до
$64\times64$ транспонированными свёртками. Один фильтр рассматривает
разные части лица одинаковым способом, поэтому DCGAN естественно
замечает локальные контуры, края и сочетания цветов.

Логит — сырая оценка проверяющего: любое число, например -3 или 2.
BCE сама обращается с ним как с вероятностной оценкой, поэтому в
лицевом дискриминаторе нет отдельного завершающего `sigmoid`.

In [ ]:
# Малые независимые начальные веса не дают всем каналам начать с
# одинаковых значений и одинаковых градиентов.
initializer = tf.keras.initializers.RandomNormal(stddev=0.02)


def dense_models():
    # Dense-генератор теряет геометрию изображения на входе: каждый
    # слой видит один вектор признаков. Последний Dense создаёт ровно
    # H×W×C чисел, а Reshape лишь меняет представление без параметров.
    generator = tf.keras.Sequential([
        tf.keras.layers.Input((NOISE_DIM,)),
        tf.keras.layers.Dense(1024, kernel_initializer=initializer),
        tf.keras.layers.LeakyReLU(0.2),
        tf.keras.layers.Dense(2048),
        tf.keras.layers.LeakyReLU(0.2),
        tf.keras.layers.Dense(IMAGE_SIZE * IMAGE_SIZE * CHANNELS, activation='tanh'),
        tf.keras.layers.Reshape((IMAGE_SIZE, IMAGE_SIZE, CHANNELS)),
    ], name='dense_generator')
    # Dense-дискриминатор сперва Flatten-ит RGB-карту. Последний Dense
    # намеренно не имеет sigmoid: BCE(from_logits=True) применит
    # численно устойчивую связку sigmoid+BCE самостоятельно.
    discriminator = tf.keras.Sequential([
        tf.keras.layers.Input((IMAGE_SIZE, IMAGE_SIZE, CHANNELS)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(2048, kernel_initializer=initializer),
        tf.keras.layers.LeakyReLU(0.2),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(1024),
        tf.keras.layers.LeakyReLU(0.2),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(1),
    ], name='dense_discriminator')
    return generator, discriminator


def dcgan_models():
    # В DCGAN вектор z сначала становится картой признаков 4×4×512.
    # Каждая Conv2DTranspose с stride=2 удваивает H и W: 4→8→16→32→64.
    # BatchNorm/ ReLU стабилизируют скрытые карты; tanh выдаёт RGB [-1,1].
    generator = tf.keras.Sequential([
        tf.keras.layers.Input((NOISE_DIM,)),
        tf.keras.layers.Dense(4 * 4 * 512, use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),
        tf.keras.layers.Reshape((4, 4, 512)),
        tf.keras.layers.Conv2DTranspose(256, 5, 2, 'same', use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),
        tf.keras.layers.Conv2DTranspose(128, 5, 2, 'same', use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),
        tf.keras.layers.Conv2DTranspose(64, 5, 2, 'same', use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU(),
        tf.keras.layers.Conv2DTranspose(CHANNELS, 5, 2, 'same', activation='tanh'),
    ], name='dcgan_generator')
    # Дискриминатор зеркально сжимает карту. Conv2D с stride=2
    # уменьшает пространственные оси 64→32→16→8→4, общие ядра
    # учитывают локальные фрагменты лица. Dropout включается только
    # при training=True во время собственной фазы D.
    discriminator = tf.keras.Sequential([
        tf.keras.layers.Input((IMAGE_SIZE, IMAGE_SIZE, CHANNELS)),
        tf.keras.layers.Conv2D(64, 5, 2, 'same'),
        tf.keras.layers.LeakyReLU(0.2),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Conv2D(128, 5, 2, 'same'),
        tf.keras.layers.LeakyReLU(0.2),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Conv2D(256, 5, 2, 'same'),
        tf.keras.layers.LeakyReLU(0.2),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Conv2D(512, 5, 2, 'same'),
        tf.keras.layers.LeakyReLU(0.2),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(1),
    ], name='dcgan_discriminator')
    return generator, discriminator


for name, factory in {'dense': dense_models, 'dcgan': dcgan_models}.items():
    # Каждый вызов factory строит новые веса: dense и dcgan не делят
    # состояние и оптимизируются как два независимых эксперимента.
    generator, discriminator = factory()
    generated = generator(tf.random.normal((config['batch_size'], NOISE_DIM)), training=False)
    logits = discriminator(generated, training=False)
    assert generated.shape == (config['batch_size'], IMAGE_SIZE, IMAGE_SIZE, CHANNELS)
    assert logits.shape == (config['batch_size'], 1)
    print(f'{name}: G={generator.count_params():,}, D={discriminator.count_params():,} параметров')

## Этап 4. Попеременный цикл и сохранение наблюдений

На шаге дискриминатора `stop_gradient` отрезает путь от синтетических
изображений к параметрам генератора. На шаге генератора
дискриминатор остаётся в графе как дифференцируемая функция, но
оптимизатор получает только параметры генератора. Это и есть
математическое содержание фразы «дискриминатор даёт сигнал, но не
обновляется».

Истории потерь не являются метрикой качества лиц сами по себе. После
каждой серии шагов сохраняется один и тот же фиксированный набор
шумовых векторов: так сетки действительно можно сравнивать по эпохам.

### Две фазы одной игры

**Фаза D — учится проверяющий.** Код берёт настоящие лица, просит G
создать синтетические и вызывает `stop_gradient`. Это как передать
проверяющему готовую фотографию: он учится на ней, но художник пока не
получает замечаний. Цели равны 0.9 для настоящих лиц и 0 для
синтетических. Оптимизатор этой фазы меняет только веса D.

**Фаза G — учится художник.** Код создаёт новый шум и больше не ставит
заглушку. D смотрит на рисунок и передаёт подсказку, что изменить,
чтобы отметка стала похожа на «настоящее», то есть на 1. Подсказка
проходит через вычисления D к пикселям и дальше к весам G. Во второй
фазе `apply_gradients` получает только веса G, поэтому D не меняется.

`training=False` у D выключает случайный Dropout, но не закрывает путь
градиента. Это настройка поведения слоёв, а не команда «не считать
производные»; именно поэтому она не равна `stop_gradient`.

В лицевых вариантах D возвращает логит $\ell$, а не готовую
вероятность. Вероятность можно мысленно получить как
$p=\sigma(\ell)=1/(1+e^{-\ell})$. Бинарная кросс-энтропия для цели
$y$ по-прежнему равна

$$
\operatorname{BCE}(y,p)=-y\log p-(1-y)\log(1-p).
$$

`BinaryCrossentropy(from_logits=True)` получает $\ell$ и выполняет
преобразование в $p$ внутри численно устойчивого вычисления. Поэтому
отдельный `sigmoid` перед этой функцией добавлять не надо.

### Потеря, производная и цепное правило

Все обучаемые числа G соберём в $\theta_G$, а все обучаемые числа D —
в $\theta_D$. Потеря — один скаляр: например, средняя ошибка всей
пачки из $B$ лиц. Если $w$ — один вес, то
$\partial\mathcal L/\partial w$ отвечает на маленький вопрос: «если
слегка увеличить именно $w$, потеря начнёт расти или убывать и как
быстро?» Все эти ответы для сети составляют градиент.

Самый простой шаг обучения имеет вид

$$
\theta\leftarrow\theta-\eta\nabla_\theta\mathcal L.
$$

Здесь $\eta$ — маленький шаг. Adam выбирает размер таких поправок с
учётом прошлых градиентов, но направление по-прежнему берёт из
$\nabla_\theta\mathcal L$.

В фазе D минимизируется

$$
\mathcal L_D=\tfrac12\left[
\operatorname{BCE}(0.9,D_{\theta_D}(\mathbf x_{\rm real}))+
\operatorname{BCE}(0,D_{\theta_D}(G_{\theta_G}(\mathbf z_D)))
\right]
$$

по весам $\theta_D$. В фазе G минимизируется

$$
\mathcal L_G=\operatorname{BCE}
\left(1,D_{\theta_D}(G_{\theta_G}(\mathbf z_G))\right)
$$

по весам $\theta_G$. Новый шум $\mathbf z$ не учится и не хранится
после шага: он лишь задаёт новый вход общей функции генератора.

Во второй формуле D остаётся частью пути вычисления. Цепное правило
говорит, что

$$
\frac{\partial\mathcal L_G}{\partial\theta_G}
=
\frac{\partial\mathcal L_G}{\partial D}
\frac{\partial D}{\partial G}
\frac{\partial G}{\partial\theta_G}.
$$

Поэтому D может объяснить G, как его пиксели повлияли на ошибку, даже
если параметры D не получают обновления в этой фазе.

### От математической формулы к TensorFlow: что делает каждый объект

Не все тензоры в `train_gan` одинаковы. `real_images`,
`discriminator_noise` и `generated_images` — временные значения
одного мини-пакета: они появляются, участвуют в вычислении и затем
больше не нужны. Параметры $\theta_G$ и $\theta_D$ — обучаемые
`tf.Variable` внутри слоёв: только их способен изменить Adam.
`d_loss` и `g_loss` — скаляры, то есть по одному числу; обратное
распространение начинается именно от такого одного числа.

### Инфографика: две фазы одного пакета

![Два независимых обновления GAN: обучение дискриминатора, затем обучение генератора](assets/diagrams/gan_two_phase_training.svg)

В фазе D подделка служит только данными для проверки: `stop_gradient`
оставляет её пиксели без изменений, но закрывает стрелку производной
к G. В фазе G D становится неподвижным измерительным прибором: через
его операции проходит замечание о качестве лица, но обновляются только
веса G.

### Внутри `GradientTape`: временный протокол прямого прохода

![Жизненный цикл GradientTape от прямого прохода до обновления Adam](assets/diagrams/tensorflow_gradient_tape.svg)

Пока выполняется `with tf.GradientTape() as tape`, TensorFlow
записывает дифференцируемые операции, связавшие входы и используемые
обучаемые `tf.Variable` со скаляром `d_loss` или `g_loss`. Tape не
является частью нейросети и сам не обновляет веса: это временная
«запись маршрута», по которой можно один раз вернуться назад.

`tape.gradient(loss, variables)` запускает этот обратный проход и
возвращает список производных в порядке `variables`. У матрицы весов
градиент — матрица той же формы, у смещения — вектор. `zip` лишь
сопоставляет пары, а `apply_gradients` передаёт их Adam. После обычного
непостоянного `gradient(...)` запись tape освобождается вместе с
сохранёнными промежуточными значениями.

### Один мини-пакет: математические объекты и TensorFlow

Здесь D возвращает логит, поэтому он может быть любым вещественным
числом. BCE внутри API сама использует $p=\sigma(\ell)$ и сводит
ошибку всего мини-пакета к одному скаляру. Ниже перечислены ровно те
объекты, которые появляются в теле `train_gan`.

| Математический объект | Имя в коде | Форма и статус | Следующее действие |
| --- | --- | --- | --- |
| Реальные лица $\mathbf x_{\rm real}$ | `real_images` | `(B,64,64,3)`, временный | Подаются в D с целью 0.9. |
| Шум для фазы D $\mathbf z_D$ | `discriminator_noise` | `(B,100)`, временный | Создаёт `fake_before_detach`. |
| Обычная подделка | `fake_before_detach` | `(B,64,64,3)`, временный | Её значения остаются, путь к G закрывается. |
| Подделка для D | `fake_images` | `(B,64,64,3)`, временный | Это `stop_gradient(...)`, цель равна 0. |
| Логиты D | `real_logits`, `fake_logits` | `(B,1)`, временные | Входы `real_loss` и `fake_loss`. |
| Потеря D | `d_loss` | скаляр $\mathcal L_D$ | `tape_d.gradient` берёт производные только по $\theta_D$. |
| Шум для G $\mathbf z_G$ | `generator_noise` | `(B,100)`, временный | Независим от шума фазы D. |
| Изображение G | `generated_images` | `(B,64,64,3)`, временный | Остаётся внутри `tape_g`. |
| Логиты D для G | `fake_logits` | `(B,1)`, временный | Сравниваются с целью 1. |
| Потеря G | `g_loss` | скаляр $\mathcal L_G$ | `tape_g.gradient` берёт производные только по $\theta_G$. |

**Это также проверка формы.** `generated_images` — изображение
`(B,64,64,3)`, а не параметр и не градиент; его оставляют внутри
`tape_g`, чтобы библиотека могла выяснить влияние каждого пикселя на
`g_loss`. Шум `(B,100)` — только вход G, поэтому оптимизатор никогда
не меняет его значения. Логиты D имеют форму `(B,1)`: по одному
неограниченному числу на каждое лицо до применения BCE.

В высоких размерностях правильную цепочку для логита $\ell$ удобнее
записывать так:

$$
\nabla_{\theta_G}\mathcal L_G
=J_{G,\theta_G}^{\mathsf T}
J_{D,\mathbf x}^{\mathsf T}
\nabla_{\ell}\mathcal L_G.
$$

$J_{G,\theta_G}$ и $J_{D,\mathbf x}$ — якобианы: таблицы локальных
влияний выходов на входы. TensorFlow не хранит гигантские таблицы;
обратный режим autodiff сразу применяет их транспонированные действия
к вектору ошибки, двигаясь от скаляра `g_loss` к параметрам. Иными
словами, вычисляется не вся таблица вопросов «каждый пиксель против
каждого веса», а один нужный ответ: как изменить каждый вес, чтобы
немного уменьшить конкретный `g_loss`.

### Шесть строк API, прочитанные как алгоритм

1. Внутри `with tf.GradientTape()` выполняется прямой проход и
   запоминается, как из входов получился `d_loss` либо `g_loss`.
2. BCE усредняет ошибки по пачке, создавая один скалярный источник
   обратного прохода.
3. `tape_d.gradient(d_loss, discriminator.trainable_variables)` либо
   `tape_g.gradient(g_loss, generator.trainable_variables)` возвращает
   градиенты в форме соответствующих весов.
4. `d_grads` и `g_grads` — списки, упорядоченные так же, как списки
   параметров; их элементы нельзя переставлять.
5. `zip` сопоставляет каждому весу именно его производную.
6. Вызов собственного `apply_gradients` меняет только параметры этого
   оптимизатора; до него в памяти находятся лишь результаты вычислений.

In [ ]:
# D возвращает логит — любое вещественное число, а не вероятность.
# from_logits=True объединяет sigmoid и BCE в устойчивую операцию.
bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)
# Эти 64 латентные точки никогда не участвуют в обучении. Они нужны
# лишь для честного сравнения картинок после разных эпох.
fixed_noise = tf.random.normal((64, NOISE_DIM), seed=SEED)


def save_grid(generator, path):
    # Режим inference исключает обновление BatchNorm и отключает
    # поведение слоёв, зависящее от обучения. Веса G не меняются.
    images = (generator(fixed_noise, training=False).numpy() + 1.0) / 2.0
    fig, axes = plt.subplots(8, 8, figsize=(8, 8))
    for axis, image in zip(axes.flat, images):
        axis.imshow(np.clip(image, 0.0, 1.0), vmin=0.0, vmax=1.0)
        axis.axis('off')
    plt.tight_layout(pad=0.05)
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)


def train_gan(name, factory):
    # Новый запуск получает собственные G, D и состояния Adam. Так
    # сравнение dense/DCGAN не наследует параметры другого варианта.
    generator, discriminator = factory()
    generator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
    discriminator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
    history = {'generator_loss': [], 'discriminator_loss': []}
    started = time.perf_counter()
    for epoch in range(1, config['epochs'] + 1):
        d_values, g_values = [], []
        for real_images in dataset:
            # real_images — NHWC-тензор (B,64,64,3) в [-1,1]. B
            # берётся из пакета, а не из config, для корректной формы
            # меток даже при изменении настройки batch.
            batch_size = tf.shape(real_images)[0]

            # === Фаза 1: обучается только дискриминатор ===
            with tf.GradientTape() as tape_d:
                # z_D имеет форму (B,100). G строит подделки (B,64,64,3)
                # в inference-режиме: статистики BatchNorm генератора
                # не получают обновления во время чужой фазы.
                discriminator_noise = tf.random.normal((batch_size, NOISE_DIM))
                fake_before_detach = generator(discriminator_noise, training=False)

                # stop_gradient сохраняет пиксели, но делает их
                # константой для tape_d. Следовательно, L_D не создаёт
                # градиента по θ_G, хотя D учится на этих изображениях.
                fake_images = tf.stop_gradient(fake_before_detach)

                # training=True включает Dropout D. В обоих случаях
                # результат — логиты формы (B,1), ещё без sigmoid.
                real_logits = discriminator(real_images, training=True)
                fake_logits = discriminator(fake_images, training=True)

                # Мишени повторяют форму логитов. 0.9 — label smoothing
                # реальных лиц, 0 — цель для подделок. BCE каждого члена
                # уже усреднена по B; среднее двух частей уравнивает их.
                real_loss = bce(tf.ones_like(real_logits) * REAL_LABEL, real_logits)
                fake_loss = bce(tf.zeros_like(fake_logits), fake_logits)
                d_loss = 0.5 * (real_loss + fake_loss)

            # tape_d вычисляет ∂L_D/∂θ_D. Список d_grads соответствует
            # D.trainable_variables по порядку и по форме. θ_G в списке
            # отсутствуют, поэтому Adam D не имеет права менять G.
            d_grads = tape_d.gradient(d_loss, discriminator.trainable_variables)
            discriminator_optimizer.apply_gradients(zip(d_grads, discriminator.trainable_variables))

            # === Фаза 2: обучается только генератор ===
            with tf.GradientTape() as tape_g:
                # Новый z_G не связан с z_D. Здесь G работает в режиме
                # training=True: BatchNorm и его обучаемые параметры
                # принадлежат именно этому обновлению.
                generator_noise = tf.random.normal((batch_size, NOISE_DIM))
                generated_images = generator(generator_noise, training=True)

                # training=False у D отключает Dropout, но НЕ обрывает
                # граф. D остаётся функцией пикселей, поэтому производная
                # L_G проходит L_G→logit_D→G(z_G) к параметрам θ_G.
                fake_logits = discriminator(generated_images, training=False)

                # Несатурирующая цель 1 просит G увеличить логиты D
                # для синтетических лиц, то есть сделать их «реальными».
                g_loss = bce(tf.ones_like(fake_logits), fake_logits)

            # Запрос только θ_G отделяет факт прохождения сигнала через
            # D от обновления его весов: dL_G/dθ_D не запрашивается и
            # discriminator_optimizer в этой фазе не вызывается.
            g_grads = tape_g.gradient(g_loss, generator.trainable_variables)
            generator_optimizer.apply_gradients(zip(g_grads, generator.trainable_variables))

            # float нужен лишь для протоколирования за пределами графа;
            # он не участвует в последующих производных.
            d_values.append(float(d_loss)); g_values.append(float(g_loss))
        # Одна точка history — средний loss всех пакетов эпохи, а не
        # случайное значение последнего мини-пакета.
        history['discriminator_loss'].append(float(np.mean(d_values)))
        history['generator_loss'].append(float(np.mean(g_values)))
        if epoch == 1 or epoch % SNAPSHOT_EVERY == 0 or epoch == config['epochs']:
            save_grid(generator, ARTIFACT_DIR / f'{name}_epoch_{epoch:03d}.png')
        print(f'{name}, эпоха {epoch:03d}: D={history["discriminator_loss"][-1]:.4f}, G={history["generator_loss"][-1]:.4f}')
    return generator, discriminator, history, time.perf_counter() - started

## Этап 5. Честное сопоставление архитектур

Обе модели получают одинаковый набор лиц, размер шума, оптимизатор и
число эпох. Сравните сетки, время работы, число параметров и динамику
обеих потерь. Не интерпретируйте меньшую потерю одной стороны как
автоматическую победу: в GAN она зависит от текущей силы соперника.

`fixed_noise` — постоянный набор карточек художника. Если лицо в одной
клетке сетки изменилось, причина в обновлении весов, а не в новом
случайном входе. Перед показом `tanh`-значения `[-1,1]` возвращаются в
`[0,1]`, потому что этого ожидает Matplotlib.

График — дневник переговоров двух игроков, а не оценка точности. Его
нужно читать вместе с сетками, конечностью чисел и разнообразием лиц.
Одинаковые лица в большинстве клеток могут означать коллапс мод, даже
если линии потерь выглядят спокойными.

In [ ]:
results = {}
for model_name, factory in {'dense': dense_models, 'dcgan': dcgan_models}.items():
    results[model_name] = train_gan(model_name, factory)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, (_, _, history, seconds) in results.items():
    axes[0].plot(history['discriminator_loss'], label=f'D, {name}')
    axes[1].plot(history['generator_loss'], label=f'G, {name}; {seconds / 60:.1f} мин')
for axis, title in zip(axes, ('Потеря дискриминатора', 'Потеря генератора')):
    axis.set(title=title, xlabel='Эпоха', ylabel='BCE по логитам')
    axis.grid(alpha=0.3); axis.legend()
plt.tight_layout(); plt.show()

for name, (_, _, history, seconds) in results.items():
    assert np.isfinite(history['generator_loss']).all()
    assert np.isfinite(history['discriminator_loss']).all()
    print(f'{name}: {seconds / 60:.1f} мин; финальные D={history["discriminator_loss"][-1]:.4f}, G={history["generator_loss"][-1]:.4f}')